In [ ]:
# fix imports
import os
import sys

module_path = os.path.abspath(os.path.join(".."))
if module_path not in sys.path:
    sys.path.append(module_path)

In [ ]:
from src.eval.hb_evaluator import HarmbenchEvaluator
from src.eval.llama_evaluator import LlamaEvaluator
from src.eval.template_evaluator import TemplateEvaluator
from src.eval.llama_guard_evaluator import LlamaGuardEvaluator
from src.eval.strong_reject_evaluator import StrongRejectEvaluator
from src.inference.configs import ServeConfig, LLMConfig


evaluators = [
    # HarmbenchEvaluator(
    #     serve_config=ServeConfig(gpu_ids=[1], startup_timeout=5 * 60, client_timeout=60),
    #     use_context=False,
    #     silent=False,
    # ),
    # LlamaGuardEvaluator(
    #     serve_config=ServeConfig(gpu_ids=[1], startup_timeout=None, client_timeout=60, verbose=True),
    #     # llm_config=LLMConfig(model_name="meta-llama/Llama-Guard-4-12B", max_model_len=4096),
    #     model_name="meta-llama/Llama-Guard-3-1B",
    #     silent=False,
    # ),
    StrongRejectEvaluator(
        serve_config=ServeConfig(gpu_ids=[1], startup_timeout=5 * 60, client_timeout=60),
        binary_thresh=0.5,
        silent=False,
    ),
    # LlamaEvaluator(
    #     serve_config=ServeConfig(gpu_ids=[1], startup_timeout=5 * 60, client_timeout=60),
    #     silent=False,
    # ),
    TemplateEvaluator(
        silent=False,
    ),
]

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from src.adver_model import AdverModel
import torch

torch.set_float32_matmul_precision("high")  # negligable effect

# model_name = "Qwen/Qwen3-0.6B"
# model_name = "meta-llama/Llama-3.2-1B-Instruct"
model_name = "meta-llama/Llama-2-7b-chat-hf"
# model_name = "lmsys/vicuna-7b-v1.5" # TODO: not instruct model, no chat template
# model_name = "mistralai/Mistral-7B-Instruct-v0.3"
# model_name = "tiiuae/falcon-7b-instruct"
# model_name = "mosaicml/mpt-7b-chat"
# model_name = "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16,
    device_map="cuda:0",
    # attn_implementation="flash_attention_2"
    # attn_implementation="sdpa",
)

if not tokenizer.pad_token:
    tokenizer.pad_token = tokenizer.eos_token

adv_model = AdverModel(
    model=model,
    tokenizer=tokenizer,
    num_tokens=20,
)

In [ ]:
import pandas as pd
from src.data import DF_Batcher

data = pd.read_csv("/home/fre.gilad/source/llm-iml/data/HarmBench/harmful_behaviors.csv")
data = data.rename(columns={"goal": "prompt"})

ds_eval = data.copy()
dl_eval = DF_Batcher(ds_eval, batch_size=25, shuffle=False)

In [ ]:
from tqdm.auto import tqdm

all_outputs = []
for batch in tqdm(dl_eval):
    convos = [[{"role": "user", "content": prompt}] for prompt in batch.prompt]
    outputs = adv_model.chat(convos, max_length=512, do_sample=False, temperature=None, top_p=None)
    all_outputs.extend(outputs)

dl_eval.set_column("response", all_outputs)

In [ ]:
eval_results = []

for evaluator in evaluators:
    print(f"Running evaluator: {evaluator.name}")
    results = evaluator.evaluate(dl_eval)
    eval_results.append(results)
    print(f"Results: {results}")

In [ ]:
# print prompts and outputs

for i, row in dl_eval.df.iterrows():
    print(f"Prompt: {row['prompt']}")
    print(f"Response: {row['response']}")
    print("-" * 80)